In [1]:
import os, json, ast
import pandas as pd
import numpy as np

OFFICIAL_BASE = "outputs/taskc/prompt_official_lastturn_rewrite_gpt_hybrid842_top5_ans_with_targets_idk_deepseek_judge.csv"

RUNS = {
    "official": {
        "alg": "outputs/taskc/prompt_official_lastturn_rewrite_gpt_hybrid842_top5_metrics_alg_raw_scores_deepseek_judge.csv",
        "faith": "outputs/taskc/prompt_official_lastturn_rewrite_gpt_hybrid842_top5_metrics_faith_raw_scores_deepseek_judge.csv",
        "rbllm": "outputs/taskc/prompt_official_lastturn_rewrite_gpt_hybrid842_top5_metrics_rbllm_raw_scores_deepseek_judge.csv",
    },
    "official_typeaware_light_v2": {
        "alg": "outputs/taskc/prompts_lastturn_rewrite_gpt_official_typeaware_light_v2_metrics_alg_raw_scores.csv",
        "faith": "outputs/taskc/prompts_lastturn_rewrite_gpt_official_typeaware_light_v2_metrics_faith_raw_scores.csv",
        "rbllm": "outputs/taskc/prompts_lastturn_rewrite_gpt_official_typeaware_light_v2_metrics_rbllm_raw_scores.csv",
    },
    "typeaware_internalcheck": {
        "alg": "outputs/taskc/prompts_lastturn_rewrite_gpt_typeaware_internalcheck_metrics_alg_raw_scores.csv",
        "faith": "outputs/taskc/prompts_lastturn_rewrite_gpt_typeaware_internalcheck_metrics_faith_raw_scores.csv",
        "rbllm": "outputs/taskc/prompts_lastturn_rewrite_gpt_typeaware_internalcheck_metrics_rbllm_raw_scores.csv",
    },
    "typeaware_balanced": {
        "alg": "outputs/taskc/prompts_lastturn_rewrite_gpt_typeaware_balanced_metrics_alg_raw_scores.csv",
        "faith": "outputs/taskc/prompts_lastturn_rewrite_gpt_typeaware_balanced_metrics_faith_raw_scores.csv",
        "rbllm": "outputs/taskc/prompts_lastturn_rewrite_gpt_typeaware_balanced_metrics_rbllm_raw_scores.csv",
    },
    "balanced": {
        "alg": "outputs/taskc/prompts_lastturn_rewrite_gpt_balanced_metrics_alg_raw_scores.csv",
        "faith": "outputs/taskc/prompts_lastturn_rewrite_gpt_balanced_metrics_faith_raw_scores.csv",
        "rbllm": "outputs/taskc/prompts_lastturn_rewrite_gpt_balanced_metrics_rbllm_raw_scores.csv",
    },
    "concise": {
        "alg": "outputs/taskc/prompts_lastturn_rewrite_gpt_concise_metrics_alg_raw_scores.csv",
        "faith": "outputs/taskc/prompts_lastturn_rewrite_gpt_concise_metrics_faith_raw_scores.csv",
        "rbllm": "outputs/taskc/prompts_lastturn_rewrite_gpt_concise_metrics_rbllm_raw_scores.csv",
    },
}

def dedup_last(df, key="task_id"):
    return df.drop_duplicates(subset=[key], keep="last").copy()

def parse_answerability(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    try:
        obj = json.loads(s)
    except Exception:
        try:
                       obj = ast.literal_eval(s)
        except Exception:
            return s.upper()
    if isinstance(obj, list) and obj:
        return str(obj[0]).upper()
    return str(obj).upper()

def get_eval_group(ans):
    ans = str(ans).upper()
    if "CONVERSATIONAL" in ans:
        return "conversational"
    if "UNANSWERABLE" in ans:
        return "unanswerable"
    if "ANSWERABLE" in ans or "PARTIAL" in ans:
        return "answerable_partial"
    return "unknown"

def hm(vals):
    vals = [float(x) for x in vals if not pd.isna(x)]
    if len(vals) == 0 or any(x <= 0 for x in vals):
        return 0.0
    return len(vals) / sum(1.0 / x for x in vals)

def hm3(a, b, c):
    if any(pd.isna(x) or x <= 0 for x in [a, b, c]):
        return 0.0
    return 3.0 / (1.0 / a + 1.0 / b + 1.0 / c)

def load_metric(path, col, new_col):
    df = dedup_last(pd.read_csv(path))
    if col not in df.columns:
        raise KeyError(f"{path} missing column {col}. Columns={list(df.columns)}")
    return df[["task_id", col]].rename(columns={col: new_col})

meta = pd.read_csv(OFFICIAL_BASE)
meta = meta[["task_id", "answerability", "domain", "Collection"]].drop_duplicates("task_id", keep="last")
meta["answerability_norm"] = meta["answerability"].apply(parse_answerability)
meta["eval_group"] = meta["answerability_norm"].apply(get_eval_group)

rows = []
merged_all = []

for name, paths in RUNS.items():
    missing = [p for p in paths.values() if not os.path.exists(p)]
    if missing:
        print(f"[SKIP] {name}, missing:")
        for p in missing:
            print("  ", p)
        continue

    alg = load_metric(paths["alg"], "RB_agg", "ALG")
    faith = load_metric(paths["faith"], "RL_F", "Faith")
    rbllm = load_metric(paths["rbllm"], "RB_llm", "RBLLM")

    df = (
        meta
        .merge(alg, on="task_id", how="left")
        .merge(faith, on="task_id", how="left")
        .merge(rbllm, on="task_id", how="left")
    )

    for c in ["ALG", "Faith", "RBLLM"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["HM3_row"] = [
        hm3(a, f, r)
        for a, f, r in zip(df["ALG"], df["Faith"], df["RBLLM"])
    ]
    df["run"] = name
    merged_all.append(df)

    main = df[df["eval_group"].isin(["answerable_partial", "unanswerable"])].copy()
    ans = df[df["eval_group"].eq("answerable_partial")].copy()
    unans = df[df["eval_group"].eq("unanswerable")].copy()

    ALG_main = main["ALG"].mean()
    Faith_main = main["Faith"].mean()
    RBLLM_main = main["RBLLM"].mean()

    rows.append({
        "run": name,
        "n_main": len(main),
        "missing_ALG": int(main["ALG"].isna().sum()),
        "missing_Faith": int(main["Faith"].isna().sum()),
        "missing_RBLLM": int(main["RBLLM"].isna().sum()),

        "ALG_main": ALG_main,
        "Faith_main": Faith_main,
        "RBLLM_main": RBLLM_main,

        "HM3_row_main": main["HM3_row"].mean(),
        "HM3_meanlevel_main": hm([ALG_main, Faith_main, RBLLM_main]),

        "ALG_answerable": ans["ALG"].mean(),
        "Faith_answerable": ans["Faith"].mean(),
        "RBLLM_answerable": ans["RBLLM"].mean(),
        "HM3_answerable": ans["HM3_row"].mean(),

        "ALG_unanswerable": unans["ALG"].mean(),
        "Faith_unanswerable": unans["Faith"].mean(),
        "RBLLM_unanswerable": unans["RBLLM"].mean(),
        "HM3_unanswerable": unans["HM3_row"].mean(),
    })

summary = pd.DataFrame(rows).sort_values("HM3_row_main", ascending=False).reset_index(drop=True)
long_df = pd.concat(merged_all, ignore_index=True) if merged_all else pd.DataFrame()

OUT_SUMMARY = "outputs/taskc/all_prompt_variants_hm3_summary_with_official_typeaware_light_v2.csv"
OUT_LONG = "outputs/taskc/all_prompt_variants_hm3_long_with_official_typeaware_light_v2.csv"

os.makedirs("outputs/taskc", exist_ok=True)
summary.to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")
long_df.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

print("========== Sorted by HM3_row_main ==========")
display(summary[[
    "run",
    "ALG_main",
    "Faith_main",
    "RBLLM_main",
    "HM3_row_main",
    "HM3_meanlevel_main",
    "HM3_answerable",
    "HM3_unanswerable",
    "missing_ALG",
    "missing_Faith",
    "missing_RBLLM",
]])

print("Saved:", OUT_SUMMARY)
print("Saved:", OUT_LONG)

========== Sorted by HM3_row_main ==========


,run,ALG_main,Faith_main,RBLLM_main,HM3_row_main,HM3_meanlevel_main,HM3_answerable,HM3_unanswerable,missing_ALG,missing_Faith,missing_RBLLM
0,official,0.449996,0.822486,0.683894,0.546318,0.612210,0.561269,0.335100,0,0,0
1,official_typeaware_light_v2,0.465026,0.802674,0.645553,0.536899,0.606635,0.564898,0.141349,0,0,0
2,typeaware_balanced,0.462923,0.795597,0.633654,0.526417,0.600570,0.556208,0.105548,0,0,0
3,typeaware_internalcheck,0.467369,0.793874,0.634976,0.523075,0.603119,0.552711,0.104394,0,0,0
4,balanced,0.439286,0.776172,0.601803,0.499844,0.574001,0.521171,0.198550,0,0,0
5,concise,0.389484,0.772125,0.528365,0.425815,0.521263,0.443971,0.169321,0,0,0


Saved: outputs/taskc/all_prompt_variants_hm3_summary_with_official_typeaware_light_v2.csv
Saved: outputs/taskc/all_prompt_variants_hm3_long_with_official_typeaware_light_v2.csv
